[Reference](https://medium.com/@han.heloir/build-stream-test-your-first-claude-managed-agent-in-30-minutes-d83fe01b7b45)

# Prerequisites and Setup

In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.7/627.7 kB 7.5 MB/s eta 0:00:00


```
pip install anthropic
```

In [2]:
import anthropic

client = anthropic.Anthropic()
# If this runs without error, you're ready
print("SDK version:", anthropic.__version__)

# Step 1: Create Your Agent

In [3]:
agent = client.beta.managed_agents.agents.create(
    name="Price Monitor",
    model="claude-sonnet-4-6",
    system="""You are a competitive pricing analyst. Given a product
    catalog CSV, research current prices for each product across
    Amazon, Best Buy, and Walmart using web search.

    For each product:
    1. Search for the exact product name + retailer
    2. Record the lowest price found at each retailer
    3. Compare against the catalog price
    4. Flag any product where the catalog price exceeds
       the lowest competitor price by more than 15%

    Output a markdown report to /mnt/session/outputs/price_report.md
    with a summary table and detailed findings per product.""",
    tools=[{"type": "agent_toolset_20260401"}]
)

print(f"Agent ID: {agent.id}")

# Step 2: Configure the Environment

In [4]:
environment = client.beta.managed_agents.environments.create(
    name="price-monitor-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"}
    }
)

print(f"Environment ID: {environment.id}")

# Step 3: Start a Session and Send Work

In [5]:
session = client.beta.managed_agents.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Price comparison run"
)

print(f"Session ID: {session.id}")

In [6]:
catalog = """product_name,sku,your_price,category
Apple AirPods Pro 2,APD-001,289.99,Audio
Sony WH-1000XM5,SNY-002,378.00,Audio
Dyson V15 Detect,DYS-003,749.99,Home
Samsung Galaxy S25 Ultra,SAM-004,1349.00,Mobile
Apple MacBook Air M4,APL-005,1249.00,Laptop
LG C4 65-inch OLED TV,LGC-006,1899.00,TV
Bose QuietComfort Ultra,BSE-007,429.99,Audio
Nintendo Switch 2,NIN-008,449.99,Gaming
Kindle Scribe 2,KDL-009,399.99,E-reader
Google Pixel 9 Pro,GPX-010,1099.00,Mobile"""

client.beta.managed_agents.sessions.events.create(
    session_id=session.id,
    events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": f"""Here is my product catalog:\n\n{catalog}\n\n
            Research competitor prices for each product across
            Amazon, Best Buy, and Walmart. Write the full analysis
            report to /mnt/session/outputs/price_report.md"""
        }]
    }]
)

# Step 4: Stream Events in Real Time

In [7]:
import json

stream = client.beta.managed_agents.sessions.events.stream(
    session_id=session.id
)

for event in stream:
    if event.type == "agent.message":
        for block in event.content:
            if block.type == "text":
                print(block.text, end="", flush=True)

    elif event.type == "agent.tool_use":
        print(f"\n🔧 [{event.name}]")

    elif event.type == "session.status_idle":
        print("\n\nAgent finished.")
        break

# Step 5: Retrieve the Output

In [8]:
files = client.beta.managed_agents.files.list(
    scope_id=session.id
)

for f in files.data:
    print(f"{f.filename} ({f.size_bytes} bytes)")

    content = client.beta.managed_agents.files.content(
        file_id=f.id
    )

    with open(f.filename, "wb") as out:
        out.write(content)

# Step 6: Test It (The Part Nobody Is Talking About)

In [9]:
rubric = """
# Competitive Price Report Rubric

## Coverage
- All 10 products from the catalog are included
- At least 2 out of 3 retailers have prices for each product
- Products are grouped by category

## Data Quality
- Each price includes the retailer name and source
- Prices are in USD with two decimal places
- The date of the price check is stated

## Analysis
- A summary table lists all products with your price,
  lowest competitor price, and percentage difference
- Products exceeding the 15% threshold are clearly flagged
- At least 3 products should be flagged as overpriced
  (given the inflated catalog prices)

## Recommendations
- Each flagged product includes a specific
  recommended price adjustment
- The total potential revenue impact is estimated

## Format
- Output is valid markdown
- The summary table is properly formatted
- Report is under 2000 words
"""

In [10]:
# Create a new session for the outcome-based run
session = client.beta.managed_agents.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Price monitor - graded run"
)

# Define the outcome (agent starts working immediately)
client.beta.managed_agents.sessions.events.create(
    session_id=session.id,
    events=[{
        "type": "user.define_outcome",
        "description": f"""Analyze this product catalog and produce
        a competitive pricing report:\n\n{catalog}""",
        "rubric": {"type": "text", "content": rubric},
        "max_iterations": 3
    }]
)

In [11]:
for event in stream:
    if event.type == "span.outcome_evaluation_start":
        print(f"\n📋 Grader evaluating (iteration {event.iteration})...")

    elif event.type == "span.outcome_evaluation_end":
        print(f"Result: {event.result}")
        print(f"Explanation: {event.explanation}")

        if event.result == "satisfied":
            print("✅ All criteria met.")
            break

In [12]:
session_status = client.beta.managed_agents.sessions.retrieve(
    session_id=session.id
)

for eval in session_status.outcome_evaluations:
    print(f"{eval.outcome_id}: {eval.result}")